In [2]:
# Import modules

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

# Import functions
from cmdstanpy import CmdStanModel
from scipy.optimize import fsolve
from tensorflow_probability.substrates import numpy as tfp
tfd = tfp.distributions

# Create stan folder if does not exists
if not os.path.exists("./stan"):
    os.mkdir("./stan")


STAN_PATH = r"C:\Users\Mohsen\.cmdstan\cmdstan-2.37.0"

from cmdstanpy import cmdstan_path, set_cmdstan_path
print(cmdstan_path())
set_cmdstan_path(STAN_PATH)
print(cmdstan_path())



C:\Users\Mohsen\.cmdstan\cmdstan-2.37.0
C:\Users\Mohsen\.cmdstan\cmdstan-2.37.0


In [71]:
doners = pd.read_csv("../../data/data_donatori_eng.csv")
data = pd.read_csv("../../data/data_eng.csv")
col_map = {"Experiment Index": "i", "Dilution Replicate Index": "k", "Control": "c", "Condition Replicate Index": "j" }
data.rename(columns=col_map, inplace=True)
doners.rename(columns=col_map, inplace=True)
data = data[~data['i'].astype(str).isin(['5','6','7', '8'])].copy()

In [72]:

for col in data.columns:
    print(col)
    print(data[col].unique())

i
['1' '2' '3' '4' '9' '10' '5,6' '7,8']
k
[1 2 3]
j
['A' 'B' 'C']
c
['No' 'Yes']
Dilution
[-1 -2 -3]
IBU
[ 6.25  0.   50.  ]
DMSO
[ 0 10]
Temperature
[28 40]
Count
['TMTC' '36' '33' '30' '4' '6' '19' '27' '29' '37' '3' '7' '200' '214'
 '178' '5' '2' '1' '0' '157' '226' '225' '11' '108' '124' '98' '8' '10'
 '165' '195' '166' '16' '13' '14' '211' '233' '210' '12' '252' '242' '198'
 '15' '186' '148' '116' '122' '127' '9' '301' '309' '315' '229' '192'
 '230' '40' '26' '35' '60' '49' '179' '140' '160' '201' '139' '106' '199'
 '163' '22' '18' '17' '170' '145' '159' '196' '213' '271' '142' '177'
 '128' '81' '129' '86' '75' '71' '306' '297' '294' '202' '239' '32' '41'
 '25' '131' '141' '221' '263' '254' '90' '168' '132' '172' '164' '20']


In [67]:
print(len(data))

369


In [68]:
for col in doners.columns:
    print(col)
    print(doners[col].unique())

i
['1' '2' '3' '4' '5,6' '7,8' '9' '10']
k
[1 2 3]
Dilution
[-6 -7 -8]
IBU
[ 6.25 50.    0.  ]
DMSO
[ 0 10]
Temperature
[28 40]
Count
[ 51  48  50   7   3   4   2   0 109 110  96  10  12  56  54  83  89  91
  15   1   5  61  43  33   6  55  40   8   9  90 106 102  17  16  13]


In [73]:
print(len(doners))
print(len(data))

72
342


In [97]:
doners

,i,k,Dilution,IBU,DMSO,Temperature,Count
0,1,1,-6,6.25,0,28,51
1,1,2,-6,6.25,0,28,48
2,1,3,-6,6.25,0,28,50
3,1,1,-7,6.25,0,28,7
4,1,2,-7,6.25,0,28,3
...,...,...,...,...,...,...,...
67,10,2,-7,0.00,10,40,16
68,10,3,-7,0.00,10,40,13
69,10,1,-8,0.00,10,40,5
70,10,2,-8,0.00,10,40,6


In [75]:
print(len(data)/len(doners))

4.75


In [152]:
rows = []
temp_i_index = '1'
temp_k_index = 1
temp_doners_t = pd.DataFrame()
data_frames = {}
for n in doners['i'].unique():
    data_frames[n] = {}
    for nn in doners[doners['i'] == n]['k'].unique():
        temp_doners = doners[(doners['i'] == n) & (doners['k'] == nn)]
        temp_data = data[(data['i'] == n) & (data['k'] == nn)]
        d_len = len(temp_doners)
        x_len = len(temp_data)
        ratio = x_len / d_len if d_len > 0 else float('nan')
        rows.append({'i': n, 'k': nn, 'doners_len': d_len, 'data_len': x_len, 'ratio': ratio})
        data_frames[n][int(nn)] = (temp_doners.copy(), temp_data.copy())
        if (n == temp_i_index) and (nn == temp_k_index):
            print("lkdsj")
            temp_doners_t = temp_doners.copy()
            
        
        

ratios_df = pd.DataFrame(rows)
print("Ratios for each (i,k):")
print(ratios_df)

lkdsj
Ratios for each (i,k):
      i  k  doners_len  data_len  ratio
0     1  1           3        18    6.0
1     1  2           3        18    6.0
2     1  3           3        18    6.0
3     2  1           3        18    6.0
4     2  2           3        18    6.0
5     2  3           3        18    6.0
6     3  1           3        18    6.0
7     3  2           3        18    6.0
8     3  3           3        18    6.0
9     4  1           3        18    6.0
10    4  2           3        18    6.0
11    4  3           3        18    6.0
12  5,6  1           3         9    3.0
13  5,6  2           3         9    3.0
14  5,6  3           3         9    3.0
15  7,8  1           3         9    3.0
16  7,8  2           3         9    3.0
17  7,8  3           3         9    3.0
18    9  1           3        12    4.0
19    9  2           3        12    4.0
20    9  3           3        12    4.0
21   10  1           3        12    4.0
22   10  2           3        12    4.0
23   10  3 

In [146]:
data_frames['1'][1][0]

,i,k,Dilution,IBU,DMSO,Temperature,Count
0,1,1,-6,6.25,0,28,51
3,1,1,-7,6.25,0,28,7
6,1,1,-8,6.25,0,28,2


In [147]:
data_frames['1'][1][1]

,i,k,j,c,Dilution,IBU,DMSO,Temperature,Count
0,1,1,A,No,-1,6.25,0,28,TMTC
3,1,1,A,No,-2,6.25,0,28,36
6,1,1,A,No,-3,6.25,0,28,4
9,1,1,A,Yes,-1,0.00,0,28,TMTC
12,1,1,A,Yes,-2,0.00,0,28,27
15,1,1,A,Yes,-3,0.00,0,28,3
162,1,1,B,No,-1,6.25,0,28,TMTC
165,1,1,B,No,-2,6.25,0,28,40
168,1,1,B,No,-3,6.25,0,28,10
171,1,1,B,Yes,-1,0.00,0,28,TMTC


In [148]:
data_frames['5,6'][1][0]

,i,k,Dilution,IBU,DMSO,Temperature,Count
36,"5,6",1,-6,50.0,10,28,61
39,"5,6",1,-7,50.0,10,28,2
42,"5,6",1,-8,50.0,10,28,0


486/63

In [149]:
data_frames['5,6'][1][1]

,i,k,j,c,Dilution,IBU,DMSO,Temperature,Count
144,"5,6",1,A,Yes,-1,0.0,10,28,TMTC
147,"5,6",1,A,Yes,-2,0.0,10,28,10
150,"5,6",1,A,Yes,-3,0.0,10,28,0
306,"5,6",1,B,Yes,-1,0.0,10,28,TMTC
309,"5,6",1,B,Yes,-2,0.0,10,28,7
312,"5,6",1,B,Yes,-3,0.0,10,28,0
432,"5,6",1,C,Yes,-1,0.0,10,28,TMTC
435,"5,6",1,C,Yes,-2,0.0,10,28,11
438,"5,6",1,C,Yes,-3,0.0,10,28,0


In [150]:
data_frames['9'][1][0]

,i,k,Dilution,IBU,DMSO,Temperature,Count
54,9,1,-6,0.0,10,28,90
57,9,1,-7,0.0,10,28,17
60,9,1,-8,0.0,10,28,5


In [151]:
data_frames['9'][1][1]

,i,k,j,c,Dilution,IBU,DMSO,Temperature,Count
108,9,1,A,No,-1,0.0,10,28,186
111,9,1,A,No,-2,0.0,10,28,1
114,9,1,A,No,-3,0.0,10,28,0
117,9,1,A,Yes,-1,0.0,0,28,116
120,9,1,A,Yes,-2,0.0,0,28,9
123,9,1,A,Yes,-3,0.0,0,28,0
270,9,1,B,No,-1,0.0,10,28,108
273,9,1,B,No,-2,0.0,10,28,3
276,9,1,B,No,-3,0.0,10,28,0
279,9,1,B,Yes,-1,0.0,0,28,86
